In [ ]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## L_CA3 — CA3 Field

**Role**: Pattern completion via recurrent attractor dynamics.
A partial cue from DG activates a partial CA3 pattern; recurrent connections reinstate the full stored pattern.

**Key parameters (Schapiro 2017 §2.a.iii, SI Table 1)**:
- `ecin_frac = 0.25`: 25% sparse ECin → CA3 direct (perforant path; TSP pathway)
- `dg_frac = 0.05`: 5% sparse mossy fibre from DG (much sparser than ECin→DG)
- `k_frac = 0.06`: ~6% active (SI Table 1) — less sparse than DG to allow overlap for completion
- `lr = 0.4`: TSP learning rate (Go reimplementation; Schapiro 2017 §2.b)
- `W_rec`: fully connected CA3→CA3 (Hopfield attractor)

**Understanding check**: What does W_rec do on the first trial of a new item?  
→ W_rec starts at zero → no recurrent input → CA3 settles based on DG + ECin direct input alone.
After the first CHL update W_rec begins to store the pattern. After several trials the attractor is stable enough that a partial DG cue reinstates the full CA3 pattern.

In [ ]:
# path & directories
import sys
from pathlib import Path

DIR_SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
DIR_VIZ = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()
sys.path.insert(0, DIR_SRC)

# hyperparameters
import torch
from layer import L_DG, L_CA3

# DG with randn*0.1 weights + use_euler=True: a single forward call (one Euler step) gives
# Vm ≈ 0.1 × net << θ=0.25, so act_DG_A / act_DG_B are likely all-zero vectors.
# CA3 cells below therefore receive no DG input; only ECin direct path drives CA3.
# This is intentional: the DG→CA3 pathway is tested separately in test_L_DG.
torch.manual_seed(42)
dg = L_DG(n_input=15, n_DG=100, k_frac=0.01, ecin_frac=0.25, use_euler=True)
dg.W.data = torch.randn(15, 100) * 0.1

a_ECin_A = torch.zeros(15); a_ECin_A[0] = 1.0; a_ECin_A[1] = 0.9
a_ECin_B = torch.zeros(15); a_ECin_B[1] = 1.0; a_ECin_B[2] = 0.9

dg.reset(); act_DG_A = dg(a_ECin_A).clone()
dg.reset(); act_DG_B = dg(a_ECin_B).clone()

print(f"DG output (act_DG_A) active units: {(act_DG_A > 0).sum().item()}  (expect 0 — single Euler step, Vm << θ)")

In [ ]:
# Instantiate and inspect structure
#
# Expected:
#   W_ff shape       : torch.Size([100, 50])  — (n_DG, n_CA3)
#   mask_ff density  : ~0.050                 — dg_frac=0.05; mossy fiber is very sparse (Schapiro 2017 §2.a.iii)
#   W_ecin shape     : torch.Size([15, 50])   — (n_ECin, n_CA3)
#   mask_ecin density: ~0.250                 — ecin_frac=0.25; perforant path direct
#   W_rec shape      : torch.Size([50, 50])   — fully connected CA3→CA3 Hopfield attractor; no mask
#   n_active target  : 3                      — k_frac=0.06 × n_CA3=50 = 3 (Schapiro 2017 SI Table 1)
ca3 = L_CA3(n_DG=100, n_ECin=15, n_CA3=50, k_frac=0.06, dg_frac=0.05, ecin_frac=0.25, use_euler=True)

print(f"W_ff shape      : {ca3.W_ff.shape}")
print(f"mask_ff shape   : {ca3.mask_ff.shape}")
print(f"mask_ff density : {ca3.mask_ff.mean():.3f}")
print(f"W_ecin shape    : {ca3.W_ecin.shape}")
print(f"mask_ecin density: {ca3.mask_ecin.mean():.3f}")
print(f"W_rec shape     : {ca3.W_rec.shape}")
print(f"n_active target : {max(1, int(ca3.k_frac * ca3.n_CA3))}")

In [ ]:
# Forward pass — sparsity check
# Uses default weights from cell-2 (uniform 0.25–0.75; Schapiro 2017 SI Table 2).
# Default weights give net input >> θ=0.25, so kWTA can select exactly k=3 winners.
# Settle for 25 cycles (one Q1 quarter) to let Vm converge (O'Reilly & Munakata 2000).
#
# Expected:
#   Active CA3 units: 3 / 50 = 0.060   — k_frac=0.06 × n_CA3=50 = 3 (Schapiro 2017 SI Table 1)
#   Note: act_DG_A is zero (see setup cell) → only ECin direct path (W_ecin @ a_ECin_A) drives CA3
ca3.reset()
for _ in range(25):
    act_ca3 = ca3(act_DG_A, a_ECin_A)

n_active = (act_ca3 > 0).sum().item()
print(f"Active CA3 units: {n_active} / {ca3.n_CA3} = {n_active / ca3.n_CA3:.3f}")
print(f"Expected ~6% = {int(ca3.k_frac * ca3.n_CA3)} units")

In [ ]:
import matplotlib.pyplot as plt

# Euler settling dynamics for CA3
# Same structure as test_L_DG euler cell: track n_active and Vm of winner unit over 50 steps.
# CA3 also receives recurrent input (W_rec @ _y); at step 0 _y=0 so W_rec contributes nothing.
# Only ECin direct path (W_ecin @ a_ECin_A) drives CA3 at the start — same as DG with W input.
#
# Expected:
#   Left panel : n_active = 0 for early steps, jumps to 3 around step ~7, stays at 3
#   Right panel: Vm rises along (1−0.9^t) curve; y jumps from 0 to ~1 when Vm crosses θ=0.25
N_STEPS = 50

ca3_euler = L_CA3(n_DG=100, n_ECin=15, n_CA3=50, k_frac=0.06, dg_frac=0.05, ecin_frac=0.25, use_euler=True)
ca3_euler.W_ff.data   = ca3.W_ff.data.clone()
ca3_euler.W_ecin.data = ca3.W_ecin.data.clone()
ca3_euler.W_rec.data  = ca3.W_rec.data.clone()

winner_idx = int((act_ca3 > 0).nonzero(as_tuple=True)[0][0])

ca3_euler.reset()
n_active_hist  = []
vm_winner_hist = []
y_winner_hist  = []

for _ in range(N_STEPS):
    out = ca3_euler(act_DG_A, a_ECin_A)
    n_active_hist.append((out > 0).sum().item())
    vm_winner_hist.append(ca3_euler._Vm[winner_idx].item())
    y_winner_hist.append(out[winner_idx].item())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(range(N_STEPS), n_active_hist, 'o-', ms=3, color='#4C72B0')
axes[0].axhline(3, color='red', linestyle='--', linewidth=1, label='target k=3')
axes[0].set_xlabel('Euler step')
axes[0].set_ylabel('Active CA3 units')
axes[0].set_title('n_active over Euler steps\n(use_euler=True, tau=0.1)')
axes[0].set_ylim(-0.5, 6)
axes[0].legend()

axes[1].plot(range(N_STEPS), vm_winner_hist, 'o-', ms=3, color='#DD8452', label='Vm (membrane potential)')
axes[1].plot(range(N_STEPS), y_winner_hist,  's--', ms=3, color='#55A868', label='y (firing rate)')
axes[1].axhline(0.25, color='gray', linestyle=':', linewidth=1.5, label='nxx1 threshold θ=0.25')
axes[1].set_xlabel('Euler step')
axes[1].set_ylabel('Value')
axes[1].set_title(f'Winner unit {winner_idx}: Vm and y\n(tau=0.1; Leabra default)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(DIR_VIZ / 'L_CA3_euler_settling.png', dpi=150, bbox_inches='tight')
plt.show()

first_active = next(t for t, n in enumerate(n_active_hist) if n > 0)
print(f"Vm crosses θ=0.25 at step {first_active}  (theory: log(0.5)/log(1-tau) ≈ {int(__import__('math').log(0.5)/__import__('math').log(1-0.1))} steps)")

In [ ]:
# Recurrent dynamics: store pattern, one CHL update, test recall from partial ECin cue
# Partial cue = ECin unit 0 only (drop unit 1 = previous-item context).
# W_rec trained on full pattern; tests whether one update is enough to complete from cue.
#
# Expected:
#   Stored pattern active units: 3         — 25-cycle settling converges to k=3
#   Overlap after 1 update:      0–2       — W_rec too weak after one update;
#                                            increases toward 3 with more updates (see next cell)

ca3.W_rec.data.zero_()

ca3.reset()
for _ in range(25):
    stored = ca3(act_DG_A, a_ECin_A)
stored = stored.clone()
print(f"Stored pattern active units: {(stored > 0).sum().item()}")

ca3.update_weights(
    a_DG_minus=torch.zeros(100), a_DG_plus=act_DG_A,
    a_ECin_minus=torch.zeros(15), a_ECin_plus=a_ECin_A,
    a_CA3_minus=torch.zeros(50), a_CA3_plus=stored,
)

# Partial cue: only ECin unit 0 active (drop unit 1 — previous item)
a_partial = torch.zeros(15); a_partial[0] = 1.0

ca3.reset()
for _ in range(25):
    recalled = ca3(act_DG_A, a_partial)
overlap = ((stored > 0) & (recalled > 0)).sum().item()
print(f"Recall from partial ECin cue after 1 CHL update:")
print(f"  active in stored  : {(stored > 0).sum().item()}")
print(f"  active in recalled: {(recalled > 0).sum().item()}")
print(f"  overlap           : {overlap}")
print("(overlap increases as W_rec strengthens — see next cell)")

In [ ]:
# Pattern completion: seed with 1/3 of stored pattern, let W_rec complete the rest
# Classic Hopfield attractor test: give partial stored pattern as initial _y,
# zero feedforward → W_rec should reinstate all 3 units after enough training.
#
# Expected:
#   Baseline overlap (W_rec=0)  : 0  — seed unit triggers no recurrence; no other units activated
#   Overlap after 30 updates    : 3  — W_rec strengthens all 3 pairwise connections;
#                                      1 seed unit is enough to reinstate the full stored pattern
#   Graph: overlap climbs from 0 toward 3 as CHL updates accumulate

ca3.W_rec.data.zero_()
ca3.reset()
for _ in range(25):
    stored = ca3(act_DG_A, a_ECin_A)
stored = stored.clone()
active_units = (stored > 0).nonzero(as_tuple=True)[0]

def recall_from_seed(ca3_layer, seed_unit, n_cycles=25):
    """Seed _y with one stored unit, zero feedforward, run settling."""
    ca3_layer.reset()
    ca3_layer._y = torch.zeros(50)
    ca3_layer._y[seed_unit] = stored[seed_unit].item()
    for _ in range(n_cycles):
        out = ca3_layer(torch.zeros(100), torch.zeros(15))
    return out

# Baseline (W_rec = 0): seed → settling → should stay at 1
recalled_base = recall_from_seed(ca3, active_units[0])
overlap_base = ((stored > 0) & (recalled_base > 0)).sum().item()

N_UPDATES = 30
overlaps = []

for _ in range(N_UPDATES):
    ca3.update_weights(
        a_DG_minus=torch.zeros(100), a_DG_plus=act_DG_A,
        a_ECin_minus=torch.zeros(15), a_ECin_plus=a_ECin_A,
        a_CA3_minus=torch.zeros(50), a_CA3_plus=stored,
    )
    recalled = recall_from_seed(ca3, active_units[0])
    overlaps.append(((stored > 0) & (recalled > 0)).sum().item())

plt.figure(figsize=(6, 3.5))
plt.axhline(overlap_base, color='orange', linestyle=':', linewidth=1.5,
            label=f'baseline (W_rec=0): {overlap_base}')
plt.plot(range(1, N_UPDATES + 1), overlaps, 'o-', color='#4C72B0', label='with W_rec')
plt.axhline(3, color='red', linestyle='--', linewidth=1, label='perfect overlap (k=3)')
plt.xlabel('CHL updates')
plt.ylabel('Overlap with stored pattern')
plt.title('Pattern completion from partial stored pattern\n(seed = 1/3 stored units; feedforward = 0)')
plt.ylim(-0.2, 3.5)
plt.legend()
plt.tight_layout()
plt.savefig(DIR_VIZ / 'L_CA3_pattern_completion.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Stored pattern active units  : {(stored > 0).sum().item()}")
print(f"Seed unit                    : {active_units[0].item()}  (1 of 3 stored units)")
print(f"Baseline overlap (W_rec=0)   : {overlap_base}")
print(f"Overlap after {N_UPDATES} updates: {overlaps[-1]}")

In [ ]:
# W_rec heatmap: before vs after CHL updates
#
# CHL update formula:
#   ΔW_rec[i, j] = lr × stored[i] × stored[j]
#
# stored has 3 active units → only positions where BOTH i and j are active get updated.
# This is Hebbian learning: "neurons that fire together, wire together."
# Result: a 3×3 block of 9 non-zero entries at the intersections of the 3 active units.
#
# Why this enables pattern completion:
#   Seed unit 3 → net = W_rec @ _y → W_rec[28,3] and W_rec[30,3] are strong
#   → units 28 and 30 receive excitatory input → kWTA selects all 3 → overlap = 3
#   One active unit "reminds" CA3 of the full stored pattern.
#
# Green lines mark the active unit positions (rows and columns 3, 28, 30).
# Red dots appear exactly at their intersections — the 9 updated positions.
#
# Expected:
#   Left  (before): all zero (flat gray)
#   Right (after) : 9 strong red entries at intersections of active units (green lines)
#                   all other positions remain zero

ca3_viz = L_CA3(n_DG=100, n_ECin=15, n_CA3=50, k_frac=0.06, dg_frac=0.05, ecin_frac=0.25, use_euler=True)
ca3_viz.W_rec.data.zero_()

ca3_viz.reset()
for _ in range(25):
    stored_viz = ca3_viz(act_DG_A, a_ECin_A)
stored_viz = stored_viz.clone()

Wrec_before = ca3_viz.W_rec.data.clone()

for _ in range(30):
    ca3_viz.update_weights(
        a_DG_minus=torch.zeros(100),  a_DG_plus=act_DG_A,
        a_ECin_minus=torch.zeros(15), a_ECin_plus=a_ECin_A,
        a_CA3_minus=torch.zeros(50),  a_CA3_plus=stored_viz,
    )

Wrec_after = ca3_viz.W_rec.data.clone()
vmax = max(Wrec_after.abs().max().item(), 1e-6)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im0 = axes[0].imshow(Wrec_before.numpy(), cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
axes[0].set_title('W_rec before CHL updates\n(all zero)', fontsize=10)
axes[0].set_xlabel('CA3 pre-synaptic unit')
axes[0].set_ylabel('CA3 post-synaptic unit')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(Wrec_after.numpy(), cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
axes[1].set_title('W_rec after 30 CHL updates\n(stored pattern units strengthened)', fontsize=10)
axes[1].set_xlabel('CA3 pre-synaptic unit')

active_units = (stored_viz > 0).nonzero(as_tuple=True)[0].tolist()
for ax in axes:
    for u in active_units:
        ax.axhline(u - 0.5, color='lime', linewidth=0.8, alpha=0.7)
        ax.axhline(u + 0.5, color='lime', linewidth=0.8, alpha=0.7)
        ax.axvline(u - 0.5, color='lime', linewidth=0.8, alpha=0.7)
        ax.axvline(u + 0.5, color='lime', linewidth=0.8, alpha=0.7)

plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
plt.suptitle(f'Active units in stored pattern (green lines): {active_units}', fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(DIR_VIZ / 'L_CA3_Wrec_update.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sanity check: mask enforcement and W_rec scope after CHL update
# This is not a novel test — mask behavior is already verified in test_L_DG.
# Purpose: confirm that L_CA3's update_weights respects the same conventions.
#
# Note: W_ff is not updated here because act_DG_A/B are zero in this test setup
# (DG outputs inactive — see setup cell). W_ff mask correctness requires active DG input.
#
# Expected:
#   W_ff  masked positions changed  : 0    — act_DG=0 → ΔW_ff=0 (not a mask test)
#   W_ff  unmasked positions changed: 0    — non-existent synapses stay zero
#   W_ecin changed at masked   positions: >0  — CHL updates existing ECin→CA3 connections
#   W_ecin changed at unmasked positions: 0   — ecin_frac=0.25; non-existent synapses stay zero
#   W_rec changed: ≤18 entries             — kWTA gives k=3 active units per phase;
#                                            outer product covers 3×3=9 positions per phase;
#                                            two different patterns → at most 9+9=18 positions change
ca3_test = L_CA3(n_DG=100, n_ECin=15, n_CA3=50, k_frac=0.06, dg_frac=0.05, ecin_frac=0.25, use_euler=True)

ca3_test.reset()
for _ in range(25):
    a_ca3_m = ca3_test(act_DG_A, a_ECin_A)
a_ca3_m = a_ca3_m.clone()

ca3_test.reset()
for _ in range(25):
    a_ca3_p = ca3_test(act_DG_B, a_ECin_B)
a_ca3_p = a_ca3_p.clone()

Wff_before    = ca3_test.W_ff.data.clone()
Wecin_before  = ca3_test.W_ecin.data.clone()
Wrec_before   = ca3_test.W_rec.data.clone()

ca3_test.update_weights(act_DG_A, act_DG_B, a_ECin_A, a_ECin_B, a_ca3_m, a_ca3_p)

dWff   = ca3_test.W_ff.data   - Wff_before
dWecin = ca3_test.W_ecin.data - Wecin_before
dWrec  = ca3_test.W_rec.data  - Wrec_before

print(f"W_ff  changed at masked   positions: {(dWff[ca3_test.mask_ff.bool()]   != 0).sum().item()}  (0: act_DG=0 → ΔW_ff=0)")
print(f"W_ff  changed at unmasked positions: {(dWff[~ca3_test.mask_ff.bool()]  != 0).sum().item()}  (expect 0)")
print(f"W_ecin changed at masked   positions: {(dWecin[ca3_test.mask_ecin.bool()]  != 0).sum().item()}")
print(f"W_ecin changed at unmasked positions: {(dWecin[~ca3_test.mask_ecin.bool()] != 0).sum().item()}  (expect 0)")
print(f"W_rec changed ({ca3_test.n_CA3}×{ca3_test.n_CA3} = {ca3_test.n_CA3**2} entries): {(dWrec != 0).sum().item()}  (≤18: k=3 active per phase → 3×3×2=18 max)")
print("W_ff and W_ecin unmasked entries must stay zero. W_rec fully updated (no mask).")